# Linear Regression 

### __Introduction__

- We will start by using this model mostly as a baseline for our other models.
- A priori, we think that this will most likely not be our best performing model, mostly because it assumes linearity of features, which may not be the case. 

## 1. Standard linear regression

In [1]:
# 1) Imports 
import numpy as np
import pandas as pd

from datetime import datetime

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# 2) Import pre processing helpers
#    -> this should define: full_train_dataset, cat_feat, num_feat,
#       basic_string_transformer, def_string_basic_transformer,
#       preprocess_categorical, MyTargetEncoder, MyOneHotEncoder, etc.
%run 05_0_preproc_helpers.ipynb  

# 3) Define target
TARGET_COL = "price"

# 4) Separate X and y from the treated dataset
y = full_train_dataset[TARGET_COL].copy()
X = full_train_dataset.drop(columns=[TARGET_COL]).copy()

# Numerical and categorical features (splitted in pre processing)
numeric_features = num_feat
categorical_features = cat_feat

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Num features:", numeric_features)
print("Cat features:", categorical_features)

X shape: (75973, 10)
y shape: (75973,)
Num features: ['year', 'mileage', 'engineSize', 'tax', 'mpg', 'previousOwners']
Cat features: ['Brand', 'model', 'transmission', 'fuelType']


In [2]:
valid_transmissions = ["MANUAL", "AUTOMATIC", "SEMIAUTO"]
valid_fueltypes    = ["PETROL", "DIESEL", "HYBRID"]

# KFold configuration
N_SPLITS = 8
RANDOM_STATE = 42

kf = KFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# Containers for overall CV metrics
cv_fold_rmses_train = []
cv_fold_maes_train  = []
cv_fold_r2s_train   = []
cv_fold_bias_train  = []

cv_fold_rmses_val = []
cv_fold_maes_val  = []
cv_fold_r2s_val   = []
cv_fold_bias_val  = []

log_path = "linreg_cv_log.txt"

with open(log_path, "w", encoding="utf-8") as log_file:

    def log(msg: str):
        """
        Write a message to the log file with timestamp.
        This way we keep track of what's happening while keeping the notebook output clean.
        """
        log_file.write(datetime.now().strftime("[%Y-%m-%d %H:%M:%S] ") + msg + "\n")
        log_file.flush()

    log("# =============================")
    log("# LINEAR REGRESSION - K-FOLD CV")
    log("# =============================")
    log(f"N_SPLITS = {N_SPLITS}")

    for fold, (train_idx, val_idx) in enumerate(kf.split(X), start=1):
        log("")
        log(f"==== FOLD {fold}/{N_SPLITS} ====")

        # 1) SEPARATE TRAIN / VALIDATION
        X_train = X.iloc[train_idx].copy()
        X_val   = X.iloc[val_idx].copy()
        y_train = y.iloc[train_idx].copy()
        y_val   = y.iloc[val_idx].copy()

        log(f"[F{fold}] X_train shape: {X_train.shape}, X_val shape: {X_val.shape}")
        log(f"[F{fold}] y_train shape: {y_train.shape}, y_val shape: {y_val.shape}")

        # NaNs before numeric imputation
        log(f"[F{fold}] NaNs before numeric imputation (num features):")
        log(str(X_train[numeric_features].isna().sum()))

        log(f"[F{fold}] NaNs before (categoricals):")
        log(str(X_train[categorical_features].isna().sum()))

        unknown_counts_before = (X_train[categorical_features] == "UNKNOWN").sum()
        log(f"[F{fold}] 'UNKNOWN' before (categoricals):")
        log(str(unknown_counts_before))

        # 2) NUMERICAL PRE PROCESSING (fit on train, transform train/val)
        year_state = fit_year_median(
            X_train,
            year_col="year",
            model_col="model"
        )
        X_train = transform_year_with_model_median(X_train, state=year_state)
        X_val   = transform_year_with_model_median(X_val,   state=year_state)

        mileage_state = fit_mileage_imputer(
            X_train,
            mileage_col="mileage",
            do_abs=True
        )
        X_train = transform_mileage_imputer(X_train, state=mileage_state)
        X_val   = transform_mileage_imputer(X_val,   state=mileage_state)

        engine_state = fit_engine_size_imputer(
            X_train,
            engine_col="engineSize"
        )
        X_train = transform_engine_size_imputer(X_train, state=engine_state)
        X_val   = transform_engine_size_imputer(X_val,   state=engine_state)

        tax_state = fit_tax_imputer(
            X_train,
            tax_col="tax",
            do_abs=True
        )
        X_train = transform_tax_imputer(X_train, state=tax_state)
        X_val   = transform_tax_imputer(X_val,   state=tax_state)

        mpg_state = fit_mpg_imputer(
            X_train,
            mpg_col="mpg",
            do_abs=True
        )
        X_train = transform_mpg_imputer(X_train, state=mpg_state)
        X_val   = transform_mpg_imputer(X_val,   state=mpg_state)

        # Paint quality was dropped
        '''
        paint_state = fit_paint_quality_imputer(
            X_train,
            paint_col="paintQuality%"
        )
        X_train = transform_paint_quality_imputer(X_train, state=paint_state)
        X_val   = transform_paint_quality_imputer(X_val,   state=paint_state)
        '''

        owners_state = fit_previous_owners_imputer(
            X_train,
            owners_col="previousOwners",
            year_col="year",
            mileage_col="mileage"
        )
        X_train = transform_previous_owners_imputer(X_train, state=owners_state)
        X_val   = transform_previous_owners_imputer(X_val,   state=owners_state)

        log(f"[F{fold}] After numeric imputation: X_train shape = {X_train.shape}, X_val shape = {X_val.shape}")
        log(f"[F{fold}] NaNs after numeric imputation (num features):")
        log(str(X_train[numeric_features].isna().sum()))

        # 3) CATEGORICAL RESOLVERS (fit on train, transform train/val)
        # Brand
        brand_state = fit_ambiguous_brand_resolver(
            train_df=X_train,
            valid_brands=valid_brands,
            brand_col="Brand",
            model_col="model",
            year_col="year",
        )

        X_train, _, brand_still_invalid_train = transform_ambiguous_brands(
            X_train,
            brand_state,
        )
        X_val, _, brand_still_invalid_val = transform_ambiguous_brands(
            X_val,
            brand_state,
        )

        log(f"[F{fold}] After solving Brand - still invalid (train): {len(brand_still_invalid_train)}")

        # Model
        model_state = fit_invalid_model_resolver(
            train_df=X_train,
            valid_models_by_brand=valid_models_by_brand,
            brand_col="Brand",
            model_col="model",
            year_col="year",
            fuel_col="fuelType",
            mpg_col="mpg",
        )

        X_train, _, model_still_invalid_train = transform_invalid_models(
            X_train,
            model_state,
        )
        X_val, _, model_still_invalid_val = transform_invalid_models(
            X_val,
            model_state,
        )

        log(f"[F{fold}] After solving model - still invalid (train): {len(model_still_invalid_train)}")

        # Transmission
        transm_state = fit_transmission_resolver(
            train_df=X_train,
            valid_transmissions=valid_transmissions,
            transm_col="transmission",
            brand_col="Brand",
            model_col="model",
            fuel_col="fuelType",
        )

        X_train, _, transm_still_invalid_train = transform_transmission_resolver(
            X_train,
            transm_state,
        )
        X_val, _, transm_still_invalid_val = transform_transmission_resolver(
            X_val,
            transm_state,
        )

        log(f"[F{fold}] After solving transmission - distinct (train): "
            + str(sorted(X_train["transmission"].astype(str).str.upper().unique())))

        # Fuel Type
        fuel_state = fit_fueltype_resolver(
            train_df=X_train,
            valid_fueltypes=valid_fueltypes,
            fuel_col="fuelType",
            brand_col="Brand",
            model_col="model",
            transm_col="transmission",
        )

        X_train, _, fuel_still_invalid_train = transform_fueltype_resolver(
            X_train,
            fuel_state,
        )
        X_val, _, fuel_still_invalid_val = transform_fueltype_resolver(
            X_val,
            fuel_state,
        )

        log(f"[F{fold}] After solving fuelType - distinct (train): "
            + str(sorted(X_train["fuelType"].astype(str).str.upper().unique())))

        # 4) SUMMARY STATS (optional)
        log(f"[F{fold}] --- NUMERICAL (train, after imputation) ---")
        log(str(X_train[numeric_features].describe().T[["mean", "std", "min", "max"]]))
        log("NaNs (train):")
        log(str(X_train[numeric_features].isna().sum()))

        log(f"[F{fold}] --- CATEGORICAL (train, after resolvers) ---")
        log("NaNs (train):")
        log(str(X_train[categorical_features].isna().sum()))
        log("'UNKNOWN' counts (train):")
        log(str((X_train[categorical_features] == "UNKNOWN").sum()))

        # 5) CATEGORICAL ENCODING
        high_card_features = ["Brand", "model"]  # target encoding
        low_card_features  = [c for c in categorical_features if c not in high_card_features]

        log(f"[F{fold}] high_card_features = {high_card_features}")
        log(f"[F{fold}] low_card_features  = {low_card_features}")

        # Target Encoding for high-card features
        te = MyTargetEncoder(smoothing=5)
        te.fit(X_train[high_card_features], y_train)

        X_train_high = te.transform(X_train[high_card_features])
        X_val_high   = te.transform(X_val[high_card_features])

        # One-Hot Encoding for low-card features
        ohe = MyOneHotEncoder()
        ohe.fit(X_train[low_card_features])

        X_train_low = ohe.transform(X_train[low_card_features])
        X_val_low   = ohe.transform(X_val[low_card_features])

        X_train_cat = pd.concat([X_train_high, X_train_low], axis=1)
        X_val_cat   = pd.concat([X_val_high,   X_val_low],   axis=1)

        log(f"[F{fold}] X_train_cat shape: {X_train_cat.shape}")
        log(f"[F{fold}] X_val_cat   shape: {X_val_cat.shape}")

        # 6) NUMERIC SCALING
        scaler = StandardScaler()
        X_train_num = scaler.fit_transform(X_train[numeric_features])
        X_val_num   = scaler.transform(X_val[numeric_features])

        X_train_num_df = pd.DataFrame(
            X_train_num,
            index=X_train.index,
            columns=[f"num_{col}" for col in numeric_features]
        )
        X_val_num_df = pd.DataFrame(
            X_val_num,
            index=X_val.index,
            columns=[f"num_{col}" for col in numeric_features]
        )

        log(f"[F{fold}] X_train_num_df shape: {X_train_num_df.shape}")
        log(f"[F{fold}] X_val_num_df   shape: {X_val_num_df.shape}")

        # 7) FINAL MATRIX (NUM + CAT)
        X_train_final = pd.concat(
            [X_train_num_df, X_train_cat],
            axis=1
        )
        X_val_final = pd.concat(
            [X_val_num_df, X_val_cat],
            axis=1
        )

        log(f"[F{fold}] X_train_final shape: {X_train_final.shape}")
        log(f"[F{fold}] X_val_final   shape: {X_val_final.shape}")

        # 8) LINEAR REGRESSION MODEL
        linreg = LinearRegression(fit_intercept=True)

        log(f"[F{fold}] Training LinearRegression...")
        linreg.fit(X_train_final, y_train)

        # Predictions
        y_pred_train = linreg.predict(X_train_final)
        y_pred_val   = linreg.predict(X_val_final)

        # 9) METRICS
        # Validation
        mse_val  = mean_squared_error(y_val, y_pred_val)
        rmse_val = np.sqrt(mse_val)
        mae_val  = mean_absolute_error(y_val, y_pred_val)
        r2_val   = r2_score(y_val, y_pred_val)
        bias_val = float(np.mean(y_pred_val - y_val))  # signed bias (pred - true)
        # If bias > 0: on average we are overpredicting; bias < 0: underpredicting

        # Train
        mse_tr  = mean_squared_error(y_train, y_pred_train)
        rmse_tr = np.sqrt(mse_tr)
        mae_tr  = mean_absolute_error(y_train, y_pred_train)
        r2_tr   = r2_score(y_train, y_pred_train)
        bias_tr = float(np.mean(y_pred_train - y_train))

        log(f"[F{fold}] TRAIN -> RMSE: {rmse_tr:.2f} | MAE: {mae_tr:.2f} | R2: {r2_tr:.4f} | Bias(pred-true): {bias_tr:.2f}")
        log(f"[F{fold}] VAL   -> RMSE: {rmse_val:.2f} | MAE: {mae_val:.2f} | R2: {r2_val:.4f} | Bias(pred-true): {bias_val:.2f}")

        # Store for global CV averages
        cv_fold_rmses_train.append(rmse_tr)
        cv_fold_maes_train.append(mae_tr)
        cv_fold_r2s_train.append(r2_tr)
        cv_fold_bias_train.append(bias_tr)

        cv_fold_rmses_val.append(rmse_val)
        cv_fold_maes_val.append(mae_val)
        cv_fold_r2s_val.append(r2_val)
        cv_fold_bias_val.append(bias_val)

    # CV SUMMARY
    log("")
    log("# ========== CV SUMMARY (LINEAR REGRESSION) ==========")
    log(f"TRAIN -> RMSE mean: {np.mean(cv_fold_rmses_train):.2f} | MAE mean: {np.mean(cv_fold_maes_train):.2f} | R2 mean: {np.mean(cv_fold_r2s_train):.4f} | Bias mean: {np.mean(cv_fold_bias_train):.2f}")
    log(f"VAL   -> RMSE mean: {np.mean(cv_fold_rmses_val):.2f} | MAE mean: {np.mean(cv_fold_maes_val):.2f} | R2 mean: {np.mean(cv_fold_r2s_val):.4f} | Bias mean: {np.mean(cv_fold_bias_val):.2f}")

print(f"Linear Regression CV finished. Detailed logs in: {log_path}")


Linear Regression CV finished. Detailed logs in: linreg_cv_log.txt


### Test and final submission

In [3]:
# 0) FINAL LINEAR REGRESSION (no hyperparams to choose, so we simply use the default)

print("Preparing final Linear Regression training...")

# 1) PREPARE FULL TRAIN FEATURES
X_full = X.copy()
y_full = y.copy()

# a) STRING NORMALIZATION 
cols_to_normalize = ["Brand", "model", "transmission", "fuelType"]

for col in cols_to_normalize:
    if col in X_full.columns:
        X_full[col] = X_full[col].apply(
            lambda x: basic_string_transformer(
                x, 
                remove_middle_spaces=True, 
                allow_extra_chars=""
            )
        )

# Define feature sets
high_card_features = ["Brand", "model"] 
low_card_features  = [c for c in categorical_features if c not in high_card_features]


# 2) NUMERIC PRE PROCESSING - FULL TRAIN (FIT & TRANSFORM)
# Year
year_state = fit_year_median(X_full, year_col="year", model_col="model")
X_full = transform_year_with_model_median(X_full, state=year_state)

# Mileage
mileage_state = fit_mileage_imputer(X_full, mileage_col="mileage", do_abs=True)
X_full = transform_mileage_imputer(X_full, state=mileage_state)

# Engine Size
engine_state = fit_engine_size_imputer(X_full, engine_col="engineSize")
X_full = transform_engine_size_imputer(X_full, state=engine_state)

# Tax
tax_state = fit_tax_imputer(X_full, tax_col="tax", do_abs=True)
X_full = transform_tax_imputer(X_full, state=tax_state)

# MPG
mpg_state = fit_mpg_imputer(X_full, mpg_col="mpg", do_abs=True)
X_full = transform_mpg_imputer(X_full, state=mpg_state)

# Paint Quality 
# paint_state = fit_paint_quality_imputer(X_full, paint_col="paintQuality%")
# X_full = transform_paint_quality_imputer(X_full, state=paint_state)

# Previous Owners
owners_state = fit_previous_owners_imputer(
    X_full, owners_col="previousOwners", year_col="year", mileage_col="mileage"
)
X_full = transform_previous_owners_imputer(X_full, state=owners_state)


# 3) CATEGORICAL RESOLVERS - FULL TRAIN (FIT & TRANSFORM)
# Brand
brand_state = fit_ambiguous_brand_resolver(
    train_df=X_full, valid_brands=valid_brands, 
    brand_col="Brand", model_col="model", year_col="year"
)
X_full, _, brand_still_invalid_full = transform_ambiguous_brands(X_full, brand_state)
print(f"Train full - Invalid Brands remaining: {len(brand_still_invalid_full)}")

# Model
model_state = fit_invalid_model_resolver(
    train_df=X_full, valid_models_by_brand=valid_models_by_brand,
    brand_col="Brand", model_col="model", year_col="year", 
    fuel_col="fuelType", mpg_col="mpg"
)
X_full, _, model_still_invalid_full = transform_invalid_models(X_full, model_state)
print(f"Train full - Invalid Models remaining: {len(model_still_invalid_full)}")

# Transmission
transm_state = fit_transmission_resolver(
    train_df=X_full, valid_transmissions=valid_transmissions,
    transm_col="transmission", brand_col="Brand", 
    model_col="model", fuel_col="fuelType"
)
X_full, _, _ = transform_transmission_resolver(X_full, transm_state)

# Fuel Type
fuel_state = fit_fueltype_resolver(
    train_df=X_full, valid_fueltypes=valid_fueltypes,
    fuel_col="fuelType", brand_col="Brand", 
    model_col="model", transm_col="transmission"
)
X_full, _, _ = transform_fueltype_resolver(X_full, fuel_state)


# 4) CATEGORICAL ENCODING - FULL TRAIN (FIT & TRANSFORM)
# Target Encoding
te = MyTargetEncoder(smoothing=5)
te.fit(X_full[high_card_features], y_full)
X_full_high = te.transform(X_full[high_card_features])

# One-Hot Encoding
ohe = MyOneHotEncoder()
ohe.fit(X_full[low_card_features])
X_full_low = ohe.transform(X_full[low_card_features])

X_full_cat = pd.concat([X_full_high, X_full_low], axis=1)
print("X_full_cat shape:", X_full_cat.shape)

# 5) NUMERIC SCALING - FULL TRAIN (FIT & TRANSFORM)
scaler = StandardScaler()
X_full_num = scaler.fit_transform(X_full[numeric_features])

X_full_num_df = pd.DataFrame(
    X_full_num,
    index=X_full.index,
    columns=[f"num_{col}" for col in numeric_features]
)
print("X_full_num_df shape:", X_full_num_df.shape)

# 6) FINAL MATRIX - FULL TRAIN
X_full_final = pd.concat([X_full_num_df, X_full_cat], axis=1)
print("X_full_final shape:", X_full_final.shape)


# 7) TRAIN FINAL LINEAR REGRESSION MODEL
linreg_final = LinearRegression(fit_intercept=True)

print("Training final Linear Regression model on full data...")
linreg_final.fit(X_full_final, y_full)
print("Done.")


# 8) PREPARE TEST FEATURES
test_df = pd.read_csv("../../project_data/test.csv")

# a) STRING NORMALIZATION (same logic as train)
for col in cols_to_normalize:
    if col in test_df.columns:
        test_df[col] = test_df[col].apply(
            lambda x: basic_string_transformer(
                x, 
                remove_middle_spaces=True, 
                allow_extra_chars=""
            )
        )

X_test = test_df[numeric_features + categorical_features].copy()

# b) NUMERIC PREPROCESSING - TEST (TRANSFORM ONLY)
X_test = transform_year_with_model_median(X_test, state=year_state)
X_test = transform_mileage_imputer(X_test, state=mileage_state)
X_test = transform_engine_size_imputer(X_test, state=engine_state)
X_test = transform_tax_imputer(X_test, state=tax_state)
X_test = transform_mpg_imputer(X_test, state=mpg_state)
# X_test = transform_paint_quality_imputer(X_test, state=paint_state)
X_test = transform_previous_owners_imputer(X_test, state=owners_state)

# c) CATEGORICAL RESOLVERS - TEST (TRANSFORM ONLY)
X_test, _, _ = transform_ambiguous_brands(X_test, brand_state)
X_test, _, _ = transform_invalid_models(X_test, model_state)
X_test, _, _ = transform_transmission_resolver(X_test, transm_state)
X_test, _, _ = transform_fueltype_resolver(X_test, fuel_state)

# d) ENCODING - TEST (TRANSFORM ONLY)
X_test_high = te.transform(X_test[high_card_features])
X_test_low  = ohe.transform(X_test[low_card_features])
X_test_cat  = pd.concat([X_test_high, X_test_low], axis=1)

# e) SCALING - TEST (TRANSFORM ONLY)
X_test_num = scaler.transform(X_test[numeric_features])
X_test_num_df = pd.DataFrame(
    X_test_num,
    index=X_test.index,
    columns=[f"num_{col}" for col in numeric_features]
)

# 9) FINAL MATRIX & PREDICTION
X_test_final = pd.concat([X_test_num_df, X_test_cat], axis=1)

# Ensure same column order as training
X_test_final = X_test_final[X_full_final.columns]

print("X_test_final shape:", X_test_final.shape)

y_test_pred = linreg_final.predict(X_test_final)

print("Predictions summary (float):")
print(pd.Series(y_test_pred).describe())

# Optional rounding
y_test_pred_round = np.round(y_test_pred).astype(int)

submission = pd.DataFrame({
    "carID": test_df["carID"].astype(int),
    "price": y_test_pred  
})

sub_name = "linear_regression_final_submission.csv"
submission.to_csv(sub_name, index=False)
print(f"Submission file created: {sub_name}")

Preparing final Linear Regression training...
Train full - Invalid Brands remaining: 0
Train full - Invalid Models remaining: 1
X_full_cat shape: (75973, 10)
X_full_num_df shape: (75973, 6)
X_full_final shape: (75973, 16)
Training final Linear Regression model on full data...
Done.
X_test_final shape: (32567, 16)
Predictions summary (float):
count    32567.000000
mean     16896.733716
std       8764.690219
min     -19396.424621
25%      10844.414724
50%      15466.170293
75%      22279.211893
max      95112.987449
dtype: float64
Submission file created: linear_regression_final_submission.csv
